# Baseline Model

## Table of Contents
1. [Model Choice](#model-choice)
2. [Feature Selection](#feature-selection)
3. [Implementation](#implementation)
4. [Evaluation](#evaluation)


In [3]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error
# Import your chosen baseline model
# Example: from sklearn.linear_model import LogisticRegression

import matplotlib.pyplot as plt
import seaborn as sns
!pip install --upgrade gdown

#url = "https://drive.google.com/file/d/1GbwrOMykQa0YJ7EXFXZxFUVIj2OFjl2c/view?usp=drive_link"
#!gdown {url}

# Распаковываем архив (название файла из ссылки обычно dataset.zip)
# Замените 'dataset.zip', если файл называется иначе после скачивания
#!unzip -q dataset.zip -d ./my_data

# usinf normal convolutional network with 2 lyers, just to seee


## Model Choice

[Explain why you've chosen a particular model as the baseline. This could be a simple statistical model or a basic machine learning model. Justify your choice.]


## Feature Selection

[Indicate which features from the dataset you will be using for the baseline model, and justify your selection.]


In [4]:
# Load the dataset
# Replace 'your_dataset.csv' with the path to your actual dataset
#df = pd.read_csv('your_dataset.csv')
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader

import torch
import torch.nn as nn


transform = transforms.Compose([
    transforms.Resize((224, 224)), # Укажите нужный вам размер
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.511, 0.488, 0.464],
                         std=[0.343, 0.321, 0.322])
])


data_path = './my_data/dataset'

my_dataset_train = torchvision.datasets.ImageFolder(root=f"{data_path}/train" , transform=transform)
my_dataset_test = torchvision.datasets.ImageFolder(root=f"{data_path}/test",  transform=transform)


print(my_dataset_train)
print(my_dataset_test)


'''
# Feature selection
# Example: Selecting only two features for a simple baseline model
X = df[['feature1', 'feature2']]
y = df['target_variable']

# Splitting the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
'''


Dataset ImageFolder
    Number of datapoints: 4720
    Root location: ./my_data/dataset/train
    StandardTransform
Transform: Compose(
               Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
               ToTensor()
               Normalize(mean=[0.511, 0.488, 0.464], std=[0.343, 0.321, 0.322])
           )
Dataset ImageFolder
    Number of datapoints: 1088
    Root location: ./my_data/dataset/test
    StandardTransform
Transform: Compose(
               Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
               ToTensor()
               Normalize(mean=[0.511, 0.488, 0.464], std=[0.343, 0.321, 0.322])
           )


"\n# Feature selection\n# Example: Selecting only two features for a simple baseline model\nX = df[['feature1', 'feature2']]\ny = df['target_variable']\n\n# Splitting the dataset\nX_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)\n"

## Implementation

[Implement your baseline model here.]



In [ ]:
# Initialize and train the baseline model
# Example for a classification problem using Logistic Regression
# model = LogisticRegression()
# model.fit(X_train, y_train)

# Your implementation code here




In [5]:
import torch
import torch.nn as nn

def train_image_classifier(model, optimizer, loss_function, dataloaders, n_epochs=10):
    # Redraw the loss plot every `PLOT_EVERY` epochs
    PLOT_EVERY = 1

    mean_train_loss_per_epoch = []
    mean_test_loss_per_epoch = []

    train_accuracy_per_epoch = []
    test_accuracy_per_epoch = []

    train_loader, test_loader = dataloaders

    for epoch in range(1, n_epochs + 1):
        model.train()

        total_train_loss_during_epoch = 0.0
        correct_predictions_training = 0

        ### START CODE HERE ###

        ### MODEL TRAINING LOOP ###
        for training_batch in train_loader:
            images, labels = training_batch

            # 1. Zero gradients
            optimizer.zero_grad()

            # 2. Forward pass
            logits = model(images)

            # 3. Compute loss
            loss = loss_function(logits,labels)

            # 4. Backward pass
            loss.backward()

            # 5. Update parameters
            optimizer.step()

            total_train_loss_during_epoch += loss.item()

            # Compare the predictions with the actual labels to get the number of correct predictions.
            correct_predictions_training += (logits.argmax(dim=1) == labels).sum().item()

        # Compute the mean loss by dividing by the number of batches (given by `len(train_loader)`).
        mean_train_loss_per_epoch.append(total_train_loss_during_epoch / len(train_loader))

        # Compute the accuracy by dividing by the number of samples (given by `len(train_loader.dataset)`).
        train_accuracy_per_epoch.append(100 * correct_predictions_training / len(train_loader.dataset))

        ### MODEL VALIDATION LOOP ###
        # Set the model to evaluation mode (disables dropout, batch norm, etc.)
        model.eval()

        total_test_loss_during_epoch = 0.0
        correct_predictions = 0

        with torch.no_grad():
            if model.training:
                raise ValueError("Model should be in evaluation mode during validation loop. Call model.eval() before this loop.")

            for test_batch in test_loader:
                images, labels = test_batch

                # The output of the model will be of shape (BATCH_SIZE, 10) with raw logits for each class.
                logits = model(images)

                # Compute loss (only needed for the live plot)
                loss   = loss_function(logits,labels)

                # To get the predicted class, we take the index of the maximum logit for each sample using `argmax`.
                predictions = logits.argmax(dim=1)
                total_test_loss_during_epoch += loss.item()

                # Compare the predictions with the actual labels to get the number of correct predictions.
                correct_predictions += (predictions == labels).sum().item()

            # Compute the mean loss by dividing by the number of batches (given by `len(test_loader)`).
            mean_test_loss_per_epoch.append(total_test_loss_during_epoch / len(test_loader))

            # Compute the accuracy by dividing by the number of samples (given by `len(test_loader.dataset)`).
            test_accuracy_per_epoch.append(100 * correct_predictions / len(test_loader.dataset))

        ### END CODE HERE ###

        ### MONITORING: LIVE PLOT OF TRAINING & TEST LOSS AND ACCURACY ###
        if epoch == 1 or epoch % PLOT_EVERY == 0:
            clear_output(wait=True)
            fig, axs = plt.subplots(1,2,figsize=(18, 4))
            axs[0].plot([1 + e for e in range(epoch)], mean_train_loss_per_epoch, label='Train Loss')
            axs[0].plot([1 + e for e in range(epoch)], mean_test_loss_per_epoch, label='Test Loss')

            axs[1].plot([1 + e for e in range(epoch)], train_accuracy_per_epoch, label='Train Accuracy')
            axs[1].plot([1 + e for e in range(epoch)], test_accuracy_per_epoch,  label='Test Accuracy')

            y_labels = ['Cross Entropy Loss', 'Accuracy (%)']
            y_scales = ['log', 'linear']
            legend_locs = ['upper right', 'lower right']
            for idx, ax in enumerate(axs):
                ax.set_xlabel('Epoch')
                ax.set_ylabel(y_labels[idx])
                ax.set_yscale(y_scales[idx])
                ax.set_xlim(1, n_epochs)
                ax.grid(which='both', color='0.95')
                ax.legend(loc=legend_locs[idx])

            # Add a centered super title to the figure.
            fig.suptitle(f'Epoch {epoch}/{n_epochs}  |  Current Accuracy on Training Set: {train_accuracy_per_epoch[-1]:.1f}%; Current Accuracy on Test Set: {test_accuracy_per_epoch[-1]:.1f}%')
            plt.tight_layout()
            plt.show()


class MyCnn(nn.Module):

    ### START CODE HERE ###

    def __init__(self, num_classes):
        # Set the random seed for reproducibility.
        torch.manual_seed(42)

'''
Then it's fed into something called MaxPool2d.
1:31
What is that? Well, let's take a look. Pooling is a common technique in convolutional neural networks
1:37
that's used to reduce the size of feature maps. It's effectively a way to throw away pixels after
1:43
a filter has been applied, compressing the data while keeping the most important parts in a way
1:49
that shouldn't affect the results. Let's see this in action. Say you had a kernel size of two for the
1:54
max pool. So on the left you have the feature map output from a filter. Since your pooling kernel
2:00
sizes two, you select a two by two area of values. You'll then only keep the largest value in this
2:06
group, in this case it's 192, and discard the rest. That's why it's called max pooling. If you repeat
2:13
this process across every two by two group of values in the feature map, you'll end up with a
2:18
smaller output, keeping only the maximum value from each two by two area. The logic here is that your
2:25
filters have already extracted the important features from the original image. So now by
2:30
applying pooling, you're compressing each filtered image, keeping just the most significant information.
2:37
As a result, less data needs to pass through the network, and the next layer sees images that were
2:43
only a quarter of the original size. And this is important because after your first convolutional
'''
        super().__init__()
        self.convolutional_blocks = nn.Sequential(
            # Conv Block 1
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1),   # Conv2d with 1 in_channel, 16 out_channels, a kernel size of 3 and a padding of 1
            nn.ReLU(),   # ReLU
            nn.MaxPool2d(kernel_size=2),   # MaxPool2d with a kernel size of 2
            # Conv Block 2
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),   # Conv2d with 16 in_channels, 32 out_channels, a kernel size of 3 and a padding of 1
            nn.ReLU(),   # ReLU
            nn.MaxPool2d(kernel_size=2),   # MaxPool2d with a kernel size of 2
        )

      # fully connected layer
      #Fully connected just means that every neuron in the input is connected to every neuron in the
      #output. And it's just another name for linear layer. In this first convolutional layer you're

        self.classifier = nn.Sequential(
            nn.Flatten(),   # Flatten
            nn.Linear(32*7*7, 128),   # Linear
            nn.ReLU(),   # ReLU
            nn.Linear(128, num_classes),   # Linear
        )

    def forward(self, x):
        x = self.convolutional_blocks(x)   # Pass image through self.convolutional_blocks
        x = self.classifier(x)             # Pass results of convolutional layers through self.classifier
        return x

    ### END CODE HERE ###





In [ ]:
'''
 classifier это «мозг» (или классификатор) вашей нейросети.
 Если сверточные слои (convolutional_blocks) выступают в роли «глаз» —
  они смотрят на картинку и находят на ней простые элементы (линии, углы, текстуры),
   то этот блок берет все найденные признаки и решает, что именно изображено на картинке
   (например, кошка, собака или машина).Давайте разберем каждую строчку внутри этого блока:

   1. nn.Flatten() — СглаживаниеНа выходе из сверточных слоев данные приходят в виде трехмерного куба:
   32 канала (карт признаков) размером 7x7 пикселей каждый.

  Компьютер видит это как многомерную матрицу.
   Полносвязный слой не умеет работать с «кубами», ему нужна одна длинная строка.Что делает:
   Вытягивает этот куб в один плоский длинный вектор.

   Результат: Вместо куба (32, 7, 7) получается длинная лента из цифр длиной
   1568 элементов (\(32 \times 7 \times 7 = 1568\)).

   2. nn.Linear(32*7*7, 128) — Первый полносвязный слойЭто классический слой
   нейросети, где каждый входной нейрон соединен с каждым выходным.Что делает: Берет наши 1568 признаков на входе и сжимает
   (перемешивает) их в 128 более сложных, абстрактных признаков. На этом этапе сеть начинает комбинировать простые детали
   (например, «круглый глаз» + «треугольное ухо») в сложные понятия.

   3. nn.ReLU() — Функция активации
   Что делает:
    Заменяет все получившиеся отрицательные числа на нули, а положительные оставляет как есть.Зачем нужна:
    Без нее нейросеть была бы просто набором простых математических уравнений и не смогла бы решать сложные нелинейные задачи
    (например, отличать сложные формы объектов).

    4. nn.Linear(128, num_classes) — Выходной слойФинальный шаг классификации.
    Что делает: Принимает 128 признаков и превращает их в итоговые ответы. Количество выходов равно num_classes
    (количеству классов в вашем датасете).Пример: Если вы классифицируете цифры от 0 до 9, то num_classes = 10.
    На выходе вы получите 10 чисел (логитов).
    Чем выше число для конкретного класса, тем больше нейросеть уверена, что на картинке именно он.
 '''

In [7]:
# Create an untrained instance of the model with fresh, randomly initialised weights and biases.
import torch.optim as optim

num_classes = len(my_dataset_train.classes)

cnn = MyCnn(num_classes)

n_params_cnn = sum(p.numel() for p in cnn.parameters())
print(f"CNN parameters: {n_params_cnn:,}")

LR = 0.01
optimizer = optim.Adam(cnn.parameters(), lr=LR)

CNN parameters: 206,952


In [8]:
loss_fn = nn.CrossEntropyLoss()


train_image_classifier(
    model         = cnn,
    optimizer     = optimizer,
    loss_function = loss_fn,
    dataloaders   = (my_dataset_train, my_dataset_test)
)

RuntimeError: mat1 and mat2 shapes cannot be multiplied (32x3136 and 1568x128)

## Evaluation

[Clearly state what metrics you will use to evaluate the model's performance. These metrics will serve as a starting point for evaluating more complex models later on.]



In [ ]:
# Evaluate the baseline model
# Example for a classification problem
# y_pred = model.predict(X_test)
# accuracy = accuracy_score(y_test, y_pred)

# For a regression problem, you might use:
# mse = mean_squared_error(y_test, y_pred)

# Your evaluation code here
